In [ ]:
import sys
sys.path.append('../')
from op_utility import reindex_structure
# Example usage of reindex_structure, which is to avoid RMSD calculation errors due to non-sequential residue numbering
source_file = 'input.pdb'
copied_file = 'output.pdb'
reindex_structure(source_file, copied_file)

## Enzyme Design Workflow

Enzyme design can be separated into two phases:

---

### 1. From-Scratch Backbone Design

In the first phase, you generate protein backbone structures *de novo*.

After designing the backbone, you should perform filtering to remove poor designs. Key criteria include:

- **Ligand SASA (Solvent Accessible Surface Area)**  
  Ensure the ligand is appropriately buried or exposed depending on your design goal.

- **Distance between three histidines and the ligand**  
  Check whether the catalytic residues (e.g., His residues) are positioned correctly relative to the ligand.

You can use the following code (provided separately) to perform this filtering step.

---

### 2. Post-Processing and Optimization

After filtering:

- **Rename your designs**  
  This is important to remove the suffix automatically added by HalluDesign.  
  Renaming ensures compatibility with downstream steps.

- **Run a second round of HalluDesign optimization**  
  Use the filtered and renamed structures as input for further refinement.

---

### Notes

- Filtering is critical to reduce computational cost in later stages.
- Proper residue–ligand geometry is essential for functional enzyme design.

In [ ]:
import pandas as pd
import numpy as np
from Bio.PDB import MMCIFParser
from tqdm import tqdm
import freesasa
import tempfile
import os

# ------------------ distance ------------------
def calc_min_dist(structure):
    zn_coords = []
    o_coords = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    
                    # ZN
                    if atom.element == "ZN" or atom.get_name().startswith("ZN"):
                        zn_coords.append(atom.get_coord())

                    # C chain O1/O5
                    if chain.id == "C" and atom.get_name() in ["O2", "O5"]:
                        o_coords.append(atom.get_coord())

    if len(zn_coords) == 0 or len(o_coords) == 0:
        return None

    zn_coords = np.array(zn_coords)
    o_coords = np.array(o_coords)

    dists = np.linalg.norm(
        zn_coords[:, None, :] - o_coords[None, :, :],
        axis=-1
    )

    return np.min(dists)


# ------------------ SASA ------------------
from Bio.PDB.SASA import ShrakeRupley

def calc_chainC_sasa_biopython(structure):

    sr = ShrakeRupley()

    sr.compute(structure, level="A")

    sasa = 0.0

    for model in structure:
        for chain in model:
            if chain.id != "C":
                continue
            for residue in chain:
                for atom in residue:
                    if hasattr(atom, "sasa"):
                        sasa += atom.sasa

    return sasa

def count_his_near_zn(structure, cutoff=3.0):

    zn_coords = []
    his_residues = set()  

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    # 找 ZN
                    if atom.element == "ZN" or atom.get_name().startswith("ZN"):
                        zn_coords.append(atom.get_coord())

    if len(zn_coords) == 0:
        return 0

    zn_coords = np.array(zn_coords)


    for model in structure:
        for chain in model:
            if chain.id != "A":
                continue

            for residue in chain:
                if residue.get_resname() != "HIS":
                    continue


                for atom in residue:
                    if atom.get_name() not in ["ND1", "NE2"]:
                        continue

                    coord = atom.get_coord()

                    dists = np.linalg.norm(zn_coords - coord, axis=1)

                    if np.any(dists <= cutoff):
                        his_residues.add((chain.id, residue.id[1]))
                        break  

    return len(his_residues)

# ------------------ Main Function ------------------
def process_csv(csv_path, out_csv):
    parser = MMCIFParser(QUIET=True)
    df = csv_path

    min_dists = []
    sasas = []
    his_counts = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing CIF"):
        cif_path = row["eval_path"]

        try:
            structure = parser.get_structure("struct", cif_path)

            dist = calc_min_dist(structure)
            sasa = calc_chainC_sasa_biopython(structure)
            his_n = count_his_near_zn(structure)

        except Exception as e:
            print(f"Error: {cif_path}, {e}")
            dist, sasa = None, None

        min_dists.append(dist)
        sasas.append(sasa)
        his_counts.append(his_n)

    df["min_ZN_O_dist"] = min_dists
    df["chainC_SASA"] = sasas
    df["ZN_HIS_count_3A"] = his_counts

    df.to_csv(out_csv, index=False)
    print(f"Saved to {out_csv}")

df1 = pd.read_csv("...../op_1/processing_results.csv")
df2 = pd.read_csv("...../op_2/processing_results.csv")
df = pd.concat([df1,df2])

df_f = df[(df["eval_A_plddt"] >= 85) & (df["eval_B_plddt"] >= 80)& (df["eval_iptm"] >= 0.80) ] # &  (df["eval_C_plddt"] >= 80) 
print(len(df))
print(len(df_f))
process_csv(df_f,"...../redesign_score.csv")


